# Démonstration des embeddings Mistral

**Objectif** : vérifier sur des exemples le comportement du modèle d'embedding `mistral-embed-2312`, utilisé pour vectoriser les événements : dimension des vecteurs, similarité entre textes, classement de vrais événements, limites.

**Rappel** : un *embedding* est un vecteur de nombres qui représente le sens d'un texte. On compare deux vecteurs avec la **similarité cosinus** (cosinus de l'angle entre eux) : proche de 1, les sens sont proches ; proche de 0, les textes sont sans rapport.

**Consignes** :
- énoncé, étape 3 : « Faites des **tests de recherche** pour vérifier l'efficacité » ;
- soutenance : Jérémy challenge sur « la **qualité des embeddings** ».

**Prérequis** : la clé `MISTRAL_API_KEY` dans le fichier `.env` (jamais affichée). **Coût** : quelques dizaines de phrases, moins d'un centime.

**Exécution** : noyau `.venv` (Python 3.12), puis « Run All ».

## 0. Préparation

La limite de `mistral-embed` en offre gratuite est de **1 requête par seconde**. La fonction `embed` envoie donc une liste de textes en **une seule requête** (batch), puis attend 1,1 s avant la suivante.

In [1]:
import sys
import time
from datetime import date
from pathlib import Path

# Le notebook s'exécute depuis notebooks/ : on ajoute la racine du projet au chemin d'import.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_mistralai import MistralAIEmbeddings

from rag import config
from rag.fetch import load_raw_events
from rag.preprocess import preprocess_events

load_dotenv(PROJECT_ROOT / ".env")  # charge MISTRAL_API_KEY (jamais affichée)

EMBEDDING_MODEL = "mistral-embed-2312"  # nom daté : reproductibilité
embedder = MistralAIEmbeddings(model=EMBEDDING_MODEL)


def embed(texts: list[str]) -> np.ndarray:
    """Vectorise une liste de textes en UNE requête (batch), puis attend 1,1 s (limite : 1 requête/s)."""
    vectors = np.array(embedder.embed_documents(texts))
    time.sleep(1.1)
    return vectors


def cosine_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Similarité cosinus entre chaque ligne de a et chaque ligne de b."""
    a_unit = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_unit = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a_unit @ b_unit.T

/Users/nadebaque/Documents/Ai Engineer/Projet_7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. À quoi ressemble un vecteur ?

On vérifie la **dimension** et la **norme** des vecteurs. S'ils sont normalisés (norme = 1), la similarité cosinus est égale au produit scalaire, ce qui guidera le choix de l'index Faiss.

In [2]:
vector = embed(["Concert de jazz au Rocher de Palmer"])[0]

print(f"Dimension               : {vector.shape[0]}")
print(f"10 premiers nombres     : {np.round(vector[:10], 4)}")
print(f"Longueur (norme)        : {np.linalg.norm(vector):.4f}")
print(f"Plus petite / plus grande valeur : {vector.min():.4f} / {vector.max():.4f}")

Dimension               : 1024
10 premiers nombres     : [-0.0104  0.0268  0.0407 -0.0003  0.0112  0.0322  0.043  -0.0233  0.0123
 -0.0459]
Longueur (norme)        : 1.0000
Plus petite / plus grande valeur : -0.1121 / 0.1074


## 2. Un sens proche sans mot commun

**Hypothèse** : pour la question « Je cherche un concert de jazz », « Soirée swing et be-bop » devrait être mieux classée qu'« Atelier peinture pour enfants », bien qu'elle ne partage aucun mot avec la question.

*À observer : le **classement** compte davantage que les valeurs elles-mêmes. Selon les modèles, les cosinus peuvent être tous assez élevés, même entre textes éloignés.*

In [3]:
example_texts = [
    "Concert de jazz",
    "Soirée swing et be-bop",
    "Comptines musicales pour les tout-petits",
    "Atelier peinture pour enfants",
]
question = "Je cherche un concert de jazz"

vectors = embed(example_texts + [question])
scores = cosine_matrix(vectors[-1:], vectors[:-1])[0]

pd.DataFrame({"Texte": example_texts, "Cosinus avec la question": scores.round(3)}).sort_values(
    "Cosinus avec la question", ascending=False
)

,Texte,Cosinus avec la question
0,Concert de jazz,0.879
1,Soirée swing et be-bop,0.784
2,Comptines musicales pour les tout-petits,0.688
3,Atelier peinture pour enfants,0.652


## 3. De vrais événements du jeu nettoyé

On prend, dans le jeu nettoyé (photo de référence), le premier événement dont le titre contient chacun de ces mots, puis on regarde quelles questions s'en rapprochent. Le texte vectorisé est « titre. description ».

In [4]:
metadata, raw_events = load_raw_events(config.REFERENCE_RAW_FILE)
events, _ = preprocess_events(raw_events, date.fromisoformat(metadata["reference_date"]))
print(f"{len(events)} événements nettoyés")


def first_event_matching(word: str) -> dict:
    """Premier événement (ordre stable du jeu nettoyé) dont le titre contient le mot donné."""
    return next(event for event in events if word.lower() in event["title"].lower())


THEMES = ["jazz", "marionnette", "exposition", "randonnée", "théâtre", "orchestre"]
chosen = [first_event_matching(word) for word in THEMES]
event_texts = [f"{event['title']}. {event['description']}" for event in chosen]

pd.DataFrame({
    "Mot cherché": THEMES,
    "Événement choisi": [event["title"] for event in chosen],
    "Ville": [event["city"] for event in chosen],
})

18771 événements nettoyés


,Mot cherché,Événement choisi,Ville
0,jazz,EXPOSITION PHOTOGRAPHIQUE « ISRAELI JAZZ » DE ...,Limoges
1,marionnette,Fabrique ta marionnette à doigts,Bordeaux
2,exposition,"Exposition ""Pollen""",Bordeaux
3,randonnée,Randonnée cyclotouriste : Convergence agglomér...,CHATELAILLON-PLAGE
4,théâtre,Atelier théâtre,Bordeaux
5,orchestre,"1 soirée, 2 orchestres !",Bordeaux


In [5]:
questions = [
    "Une sortie en famille avec de jeunes enfants",
    "Écouter de la musique classique",
    "Faire une balade dans la nature",
    "Voir une pièce de théâtre",
]

vectors = embed(questions + event_texts)
scores = cosine_matrix(vectors[:len(questions)], vectors[len(questions):])

table = pd.DataFrame(scores.round(3), index=questions, columns=[event["title"][:35] for event in chosen])
table.style.background_gradient(axis=1, cmap="Greens")  # plus la case est foncée, plus le sens est proche

,EXPOSITION PHOTOGRAPHIQUE « ISRAELI,Fabrique ta marionnette à doigts,"Exposition ""Pollen""",Randonnée cyclotouriste : Convergen,Atelier théâtre,"1 soirée, 2 orchestres !"
Une sortie en famille avec de jeunes enfants,0.651000,0.744000,0.682000,0.662000,0.666000,0.713000
Écouter de la musique classique,0.659000,0.668000,0.649000,0.619000,0.662000,0.723000
Faire une balade dans la nature,0.663000,0.699000,0.696000,0.695000,0.678000,0.718000
Voir une pièce de théâtre,0.662000,0.731000,0.685000,0.651000,0.790000,0.763000


In [6]:
# Événement le plus proche de chaque question
for question_text, row in zip(questions, scores):
    best = chosen[int(row.argmax())]
    print(f"{question_text:45} -> {best['title']} ({row.max():.3f})")

Une sortie en famille avec de jeunes enfants  -> Fabrique ta marionnette à doigts (0.744)
Écouter de la musique classique               -> 1 soirée, 2 orchestres ! (0.723)
Faire une balade dans la nature               -> 1 soirée, 2 orchestres ! (0.718)
Voir une pièce de théâtre                     -> Atelier théâtre (0.790)


## 4. Les limites

**Hypothèse** : un embedding résume le **thème** d'un texte, mais distingue mal des textes qui ne diffèrent que par un **fait précis** (date, ville, gratuité) ou par une **négation**. Si c'est confirmé, la recherche sémantique devra être complétée par des **filtres sur les métadonnées** (dates, ville). On compare des paires de phrases.

In [7]:
pairs = [
    ("Concert de jazz", "Soirée swing et be-bop"),                                # sens proche, mots différents
    ("Concert de jazz", "Atelier de recherche d'emploi"),                         # sens éloigné
    ("Concert le 12 novembre à Bordeaux", "Concert le 21 novembre à Bordeaux"),   # seule la date change
    ("Concert gratuit", "Concert payant"),                                        # information pratique opposée
    ("Concert gratuit", "Concert non gratuit"),                                   # négation
    ("Spectacle de danse à Pau", "Spectacle de danse à Limoges"),                 # seule la ville change
]

vectors = embed([text for pair in pairs for text in pair])
rows = []
for i, (text_a, text_b) in enumerate(pairs):
    score = cosine_matrix(vectors[2 * i:2 * i + 1], vectors[2 * i + 1:2 * i + 2])[0, 0]
    rows.append({"Texte A": text_a, "Texte B": text_b, "Cosinus": round(float(score), 3)})

pd.DataFrame(rows)

,Texte A,Texte B,Cosinus
0,Concert de jazz,Soirée swing et be-bop,0.848
1,Concert de jazz,Atelier de recherche d'emploi,0.672
2,Concert le 12 novembre à Bordeaux,Concert le 21 novembre à Bordeaux,0.968
3,Concert gratuit,Concert payant,0.849
4,Concert gratuit,Concert non gratuit,0.948
5,Spectacle de danse à Pau,Spectacle de danse à Limoges,0.870


## 5. Synthèse

| Point vérifié | Résultat | Conséquence |
|---|---|---|
| Dimension, norme | 1 024 dimensions ; norme = 1,0000 (vecteurs normalisés) | Cosinus = produit scalaire : guide le choix de l'index Faiss |
| Sens proche sans mot commun | « Soirée swing et be-bop » (0,784) devant « Atelier peinture pour enfants » (0,652) : hypothèse confirmée | La recherche sémantique fonctionne |
| Échelle des scores | ≈ 0,65 à 0,67 entre textes sans rapport (« Concert de jazz » / « Atelier de recherche d'emploi » : 0,672) ; ≈ 0,85 à 0,88 pour des sens proches | Interpréter l'écart au bruit de fond, pas la valeur absolue ; un éventuel seuil de similarité se calibre sur des exemples |
| Vrais événements | 3 questions sur 4 bien classées ; « Faire une balade dans la nature » → « 1 soirée, 2 orchestres ! » (0,718), avec des scores tous compris entre 0,66 et 0,72 | Aucun des 6 candidats ne parle de nature : la recherche renvoie toujours le résultat le moins mauvais. Prévoir un seuil, ou un prompt qui autorise « je n'ai pas trouvé » |
| Qualité du texte vectorisé | Textes pauvres (« Randonnée cyclotouriste… Organisation : CODEP CHARENTE-MARITIME ») ou génériques (« Deux expériences orchestrales à vivre… », proche des 4 questions : 0,71 à 0,76) | Enrichir le texte vectorisé (description longue, mots-clés, lieu) |
| Limites | Date différente 0,968 ; « gratuit » / « non gratuit » 0,948 ; Pau / Limoges 0,870 : plus proches que de vrais synonymes (jazz / swing : 0,848) | Hypothèse confirmée : filtres sur les métadonnées (dates, ville) indispensables |